# Simulating the Amplitude-Based Collisionless QLBM with parallel initial conditions

Marker qubits can also index *initial conditions*: each marker state seeds a different
population, and the single simulation evolves all of them side by side over the same geometry.

`qlbm` builds its circuits as `qarp` blocks and simulates them with `qarp`'s statevector simulator.
A run is described by a `SimulationConfig`, which ties the four algorithmic circuits — initial
conditions, time step algorithm, postprocessing, and measurement — to a compiler optimization level
and a shot budget. `cfg.validate()` type-checks the configuration and `cfg.prepare_for_simulation()`
lowers every component into a built `qarp` block. A `QarpRunner` then drives the time loop and writes
one ParaView frame per time step into `output_dir/paraview`.

`statevector_snapshots=True` is the fast path: the runner hands the state produced at step $k$ back to
the simulator as the *initial state* of step $k+1$, so simulating $n$ steps costs $n$ applications of
the algorithm instead of $\frac{n(n+1)}{2}$. Passing `qarp.EXACT` instead of a shot count returns exact
outcome probabilities rather than sampled counts.

> **Demo-sized.** The lattice below is a 16x16 D2Q9 domain with one bounce-back obstacle
> and two parallel initial conditions, simulated for 10 time steps. Simulation cost grows exponentially in the number of qubits, so scale the lattice and
> `NUM_STEPS` up deliberately.

In [ ]:
from qlbm.components import (
    CQLBM,
    ABGridMeasurement,
    ABParallelDiscreteUniformInitialConditions,
    EmptyPrimitive,
)
from qlbm.infra import QarpRunner, SimulationConfig
from qlbm.lattice import ABLattice
from qlbm.tools.utils import create_directory_and_parents

In [ ]:
lattice = ABLattice(
    {
        "lattice": {"dim": {"x": 16, "y": 16}, "velocities": "d2q9"},
        "geometry": [
            {"shape": "cuboid", "x": [10, 13], "y": [6, 14], "boundary": "bounceback"},
        ],
    }
)

output_dir = "qlbm-output/ab-pic-d2q9-16x16-1-obstacle-qarp"
create_directory_and_parents(output_dir)

In [ ]:
# Set the number of marker registers, such that we can have up to 2 parallel initial conditions
lattice.set_num_marker_qubits(1)

In [ ]:
# ICs moving E and NE, assembled on a rectangle [0, 1, 2, 3] x [0, ..., 15] on marker |0>
# and W and SW, assembled on a rectangle [0, ..., 15] x [0, 1, 2, 3] on marker |1>
ics = ABParallelDiscreteUniformInitialConditions(
    lattice,
    [[1, 5], [3, 7]],
    [([0, 1], [0, 1, 2, 3]), ([0, 1, 2, 3], [0, 1])],
)

ics.plot()

In [ ]:
# Number of shots to sample for each time step
NUM_SHOTS = 2**12

# Number of time steps to simulate
NUM_STEPS = 10

In [ ]:
cfg = SimulationConfig(
    initial_conditions=ics,
    algorithm=CQLBM(lattice),
    postprocessing=EmptyPrimitive(lattice),
    measurement=ABGridMeasurement(lattice),
    optimization_level=0,
    shots=NUM_SHOTS,
)

cfg.validate()
cfg.prepare_for_simulation()

In [ ]:
runner = QarpRunner(cfg, lattice, seed=1234)


# Snapshots carry the statevector between time steps, making the loop O(NUM_STEPS)
runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)